In [13]:
# math_datasets_eda_clean.py (v7 – **complete**)
"""Exploratory Data Analysis for several math‑reasoning datasets.

* **v5 fixes**
  * The previous upload was truncated half‑way through `summarise()` – now the
    script is fully intact and runnable end‑to‑end.
  * Extra parenthesis in `analyze_mathinstruct()` removed.
  * Complexity‑ratio section skips datasets that lack a second field (e.g.
    OpenWebMath) to avoid division by zero/NaN.
* **v6 additions**
  * Added FineWeb dataset analysis
* **v7 enhancements**
  * Increased font sizes for all plots for better readability
  * Standardized plot styling
"""
from __future__ import annotations

import json
import xml.etree.ElementTree as ET
from collections import Counter
from pathlib import Path
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import nltk
import numpy as np
import pandas as pd
import seaborn as sns
from datasets import load_dataset
from nltk.tokenize import word_tokenize

###############################################################################
# Configuration
###############################################################################

ASDIV_PATH = Path(
    "/Users/jonathan/Library/Mobile Documents/com~apple~CloudDocs/Master/Master Thesis/math-reasoning-in-language-models/data/curriculum_learning/1_ASDiv/ASDiv.xml"
)
PARAMAWPS_PATH = Path(
    "/Users/jonathan/Library/Mobile Documents/com~apple~CloudDocs/Master/Master Thesis/math-reasoning-in-language-models/data/curriculum_learning/2_ParaMAWPS/ParaMAWPS_trainset.json"
)
DMATH_PATH = Path(
    "/Users/jonathan/Library/Mobile Documents/com~apple~CloudDocs/Master/Master Thesis/math-reasoning-in-language-models/data/curriculum_learning/4_Dmath/dmath_train.json"
)

LOCAL_PLOT_DIR = Path("domain_plots")
CLOUD_PLOT_DIR = Path(
    "/Users/jonathan/Library/Mobile Documents/com~apple~CloudDocs/Master/Master Thesis/math-reasoning-in-language-models/plots/plots_images/eda"
)

SAMPLE_SIZE_OPENWEBMATH = 10_000
SAMPLE_SIZE_MATHINSTRUCT = 10_000
SAMPLE_SIZE_FINEWEB = 10_000

# Font sizes for better readability
TITLE_FONTSIZE = 18
LABEL_FONTSIZE = 14
TICK_FONTSIZE = 12
LEGEND_FONTSIZE = 12
SUPTITLE_FONTSIZE = 20

###############################################################################
# Initialisation
###############################################################################

nltk.download("punkt", quiet=True)
plt.style.use("ggplot")
sns.set(style="whitegrid")

# Set global font sizes
plt.rcParams['font.size'] = TICK_FONTSIZE
plt.rcParams['axes.titlesize'] = TITLE_FONTSIZE
plt.rcParams['axes.labelsize'] = LABEL_FONTSIZE
plt.rcParams['xtick.labelsize'] = TICK_FONTSIZE
plt.rcParams['ytick.labelsize'] = TICK_FONTSIZE
plt.rcParams['legend.fontsize'] = LEGEND_FONTSIZE
plt.rcParams['figure.titlesize'] = SUPTITLE_FONTSIZE

for _dir in (LOCAL_PLOT_DIR, CLOUD_PLOT_DIR):
    _dir.mkdir(parents=True, exist_ok=True)

###############################################################################
# Helpers
###############################################################################

def count_tokens(text: str | None) -> int:
    return len(word_tokenize(text)) if isinstance(text, str) else 0


def _savefig(fig: plt.Figure, name: str) -> None:
    fig.savefig(LOCAL_PLOT_DIR / name, bbox_inches="tight", dpi=300)
    fig.savefig(CLOUD_PLOT_DIR / name, bbox_inches="tight", dpi=300)


def _barplot_counter(counter: Counter, filename: str, title: str, *, rotate: bool = True) -> None:
    labels, counts = zip(*counter.most_common(10))
    fig, ax = plt.subplots(figsize=(12, 7))  # Slightly larger figure
    bars = sns.barplot(x=list(counts), y=list(labels), ax=ax)
    
    # Add value annotations to bars
    for i, p in enumerate(bars.patches):
        width = p.get_width()
        ax.text(width + 0.3, p.get_y() + p.get_height()/2, 
                f'{int(width)}', ha='left', va='center', fontsize=TICK_FONTSIZE-1)
    
    ax.set_xlabel("Count", fontsize=LABEL_FONTSIZE)
    if rotate:
        ax.set_yticklabels(labels, fontsize=TICK_FONTSIZE)
    
    # Add some padding to the right to accommodate annotations
    plt.tight_layout()
    ax.set_xlim(right=ax.get_xlim()[1] * 1.15)
    
    _savefig(fig, filename)
    plt.close(fig)

###############################################################################
# Dataset‑specific analyses
###############################################################################

def analyze_openwebmath(sample_size: int) -> Tuple[List[int], List[int]]:
    print("\nLoading OpenWebMath …")
    ds = load_dataset("open-web-math/open-web-math", split=f"train[:{sample_size}]")
    print(f"Loaded {len(ds):,} examples")

    text_tokens, domains, sources = [], [], []
    for row in ds:
        text_tokens.append(count_tokens(row.get("text", "")))
        url = row.get("url", "")
        if url:
            parts = url.split("/")
            if len(parts) > 2:
                domains.append(parts[2])
        if src := row.get("source", ""):
            sources.append(src)

    fig, ax = plt.subplots(figsize=(10, 6))
    sns.histplot(text_tokens, kde=True, ax=ax)
    ax.set_xlabel("Tokens", fontsize=LABEL_FONTSIZE)
    ax.set_ylabel("Count", fontsize=LABEL_FONTSIZE)
    _savefig(fig, "openwebmath_tokens.png")
    plt.close(fig)

    if domains:
        _barplot_counter(Counter(domains), "openwebmath_domains.png", "Top Domains – OpenWebMath")
    if sources:
        _barplot_counter(Counter(sources), "openwebmath_sources.png", "Top Sources – OpenWebMath")

    return text_tokens, []  # second list empty indicates single‑field dataset


def analyze_fineweb(sample_size: int) -> Tuple[List[int], List[int]]:
    print("\nLoading FineWeb …")
    # Use a streaming approach to avoid downloading the entire dataset
    ds = load_dataset("HuggingFaceFW/fineweb", "sample-10BT", split="train", streaming=True)
    # Take only the requested sample size
    ds = ds.take(sample_size)
    ds = list(ds)  # Materialize the iterator
    print(f"Loaded {len(ds):,} examples")

    text_tokens, domains = [], []
    for row in ds:
        text_tokens.append(count_tokens(row.get("text", "")))
        url = row.get("url", "")
        if url:
            parts = url.split("/")
            if len(parts) > 2:
                domains.append(parts[2])
        # If the dataset has a token_count field, we could also analyze that separately

    fig, ax = plt.subplots(figsize=(10, 6))
    sns.histplot(text_tokens, kde=True, ax=ax, bins=30)
    ax.set_xlabel("Tokens", fontsize=LABEL_FONTSIZE)
    ax.set_ylabel("Count", fontsize=LABEL_FONTSIZE)
    _savefig(fig, "fineweb_tokens.png")
    plt.close(fig)

    if domains:
        _barplot_counter(Counter(domains), "fineweb_domains.png", "Top Domains – FineWeb")

    return text_tokens, []  # second list empty indicates single‑field dataset


def analyze_asdiv() -> Tuple[List[int], List[int]]:
    print("\nLoading ASDiv …")
    root = ET.parse(ASDIV_PATH).getroot()

    body_t, q_t, a_t, sol_types, grades = [], [], [], [], []
    for prob in root.findall(".//Problem"):
        body_t.append(count_tokens(prob.findtext("Body", "")))
        q_t.append(count_tokens(prob.findtext("Question", "")))
        a_t.append(count_tokens(prob.findtext("Answer", "")))
        sol_types.append(prob.findtext("Solution-Type", ""))
        grades.append(prob.get("Grade", ""))

    fig, ax = plt.subplots(figsize=(12, 7))
    sns.histplot([b + q for b, q in zip(body_t, q_t)], bins=30, label="Problem", color="steelblue")
    sns.histplot(a_t, bins=30, label="Answer", color="salmon")
    ax.set_xlabel("Tokens", fontsize=LABEL_FONTSIZE)
    ax.set_ylabel("Count", fontsize=LABEL_FONTSIZE)
    ax.legend(fontsize=LEGEND_FONTSIZE)
    _savefig(fig, "asdiv_tokens.png")
    plt.close(fig)

    _barplot_counter(Counter(sol_types), "asdiv_solution_types.png", "Solution Types – ASDiv")
    _barplot_counter(Counter(grades), "asdiv_grades.png", "Grade Distribution – ASDiv", rotate=False)

    return [b + q for b, q in zip(body_t, q_t)], a_t


def analyze_paramawps() -> Tuple[List[int], List[int]]:
    print("\nLoading ParaMAWPS …")
    data = json.loads(PARAMAWPS_PATH.read_text())

    text_t, eq_t, ops = [], [], []
    for item in data:
        text_t.append(count_tokens(item.get("original_text") or item.get("segmented_text", "")))
        eq = item.get("equation", "")
        eq_t.append(count_tokens(eq))
        if "+" in eq:
            ops.append("addition")
        elif "-" in eq and "*" not in eq:
            ops.append("subtraction")
        elif "*" in eq and "/" not in eq:
            ops.append("multiplication")
        elif "/" in eq:
            ops.append("division")
        else:
            ops.append("other")

    fig, ax = plt.subplots(figsize=(12, 7))
    sns.histplot(text_t, bins=30, label="Problem text", color="steelblue")
    sns.histplot(eq_t, bins=30, label="Equation", color="salmon")
    ax.set_xlabel("Tokens", fontsize=LABEL_FONTSIZE)
    ax.set_ylabel("Count", fontsize=LABEL_FONTSIZE)
    ax.legend(fontsize=LEGEND_FONTSIZE)
    _savefig(fig, "paramawps_tokens.png")
    plt.close(fig)

    _barplot_counter(Counter(ops), "paramawps_operations.png", "Equation Operation Types – ParaMAWPS")
    return text_t, eq_t


def analyze_dmath() -> Tuple[List[int], List[int]]:
    print("\nLoading DMath …")
    data = json.loads(DMATH_PATH.read_text())

    q_t, a_t, cats = [], [], []
    for item in data.values():
        q_t.append(count_tokens(item.get("question_en", "")))
        a_t.append(count_tokens(item.get("answer_en", "")))
        cats.append(item.get("category", "Unknown"))

    fig, ax = plt.subplots(figsize=(12, 7))
    sns.histplot(q_t, bins=30, label="Question", color="steelblue")
    sns.histplot(a_t, bins=30, label="Answer", color="salmon")
    ax.set_xlabel("Tokens", fontsize=LABEL_FONTSIZE)
    ax.set_ylabel("Count", fontsize=LABEL_FONTSIZE)
    ax.legend(fontsize=LEGEND_FONTSIZE)
    _savefig(fig, "dmath_tokens.png")
    plt.close(fig)

    _barplot_counter(Counter(cats), "dmath_categories.png", "Top Categories – DMath")
    return q_t, a_t


def analyze_mathinstruct(sample_size: int) -> Tuple[List[int], List[int]]:
    print("\nLoading MathInstruct …")
    ds = load_dataset("TIGER-Lab/MathInstruct", split=f"train[:{sample_size}]")
    print(f"Loaded {len(ds):,} examples")

    instr_t, out_t, sources = [], [], []
    for row in ds:
        instr_t.append(count_tokens(row.get("instruction", "")))
        out_t.append(count_tokens(row.get("output", "")))
        if src := row.get("source", ""):
            sources.append(src)

    fig, ax = plt.subplots(1, 2, figsize=(15, 7))
    sns.histplot(instr_t, kde=True, ax=ax[0], bins=30)
    ax[0].set_xlabel("Tokens", fontsize=LABEL_FONTSIZE)
    ax[0].set_ylabel("Count", fontsize=LABEL_FONTSIZE)
    
    sns.histplot(out_t, kde=True, ax=ax[1], bins=30)
    ax[1].set_xlabel("Tokens", fontsize=LABEL_FONTSIZE)
    ax[1].set_ylabel("Count", fontsize=LABEL_FONTSIZE)
    
    fig.suptitle("MathInstruct Token Counts", fontsize=SUPTITLE_FONTSIZE, y=1.02)
    plt.tight_layout()
    _savefig(fig, "mathinstruct_tokens.png")
    plt.close(fig)

    if sources:
        _barplot_counter(Counter(sources), "mathinstruct_sources.png", "Top Sources – MathInstruct")

    return instr_t, out_t

###############################################################################
# Summary / comparison
###############################################################################

def summarise(datasets: Dict[str, Tuple[List[int], List[int]]]) -> None:
    # Table ------------------------------------------------------------------
    rows = []
    for name, (field1, field2) in datasets.items():
        rows.append(
            {
                "Dataset": name,
                "Field 1 Tokens (Avg)": f"{np.mean(field1):.2f}" if field1 else "NA",
                "Field 2 Tokens (Avg)": f"{np.mean(field2):.2f}" if field2 else "NA",
                "Samples": len(field1) or len(field2),
            }
        )
    df = pd.DataFrame(rows)
    print("\n=== Token‑count summary ===")
    print(df.to_string(index=False))

    # Bar plot ---------------------------------------------------------------
    fig, ax = plt.subplots(figsize=(14, 8))
    x = np.arange(len(df))
    width = 0.35
    field1_vals = pd.to_numeric(df["Field 1 Tokens (Avg)"], errors="coerce")
    field2_vals = pd.to_numeric(df["Field 2 Tokens (Avg)"], errors="coerce")
    
    bars1 = ax.bar(x - width / 2, field1_vals, width, label="Field 1", color="steelblue")
    bars2 = ax.bar(x + width / 2, field2_vals, width, label="Field 2", color="salmon")
    
    # Add value labels on top of bars
    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            if not np.isnan(height) and height > 0:
                ax.text(bar.get_x() + bar.get_width()/2., height + 0.1,
                        f'{height:.1f}', ha='center', va='bottom', fontsize=TICK_FONTSIZE)
    
    ax.set_xticks(x)
    ax.set_xticklabels(df["Dataset"], fontsize=LABEL_FONTSIZE)
    ax.set_ylabel("Average Token Count", fontsize=LABEL_FONTSIZE)
    ax.set_xlabel("Dataset", fontsize=LABEL_FONTSIZE)
    ax.legend(fontsize=LEGEND_FONTSIZE)
    
    # Add grid for better readability
    ax.grid(axis='y', linestyle='--', alpha=0.7)
    
    plt.tight_layout()
    _savefig(fig, "token_comparison.png")
    plt.close(fig)

###############################################################################
# Main
###############################################################################

def main() -> None:
    datasets: Dict[str, Tuple[List[int], List[int]]] = {}

    try:
        datasets["OpenWebMath"] = analyze_openwebmath(SAMPLE_SIZE_OPENWEBMATH)
    except Exception as e:
        print(f"WARNING: OpenWebMath failed: {e}")
        
    try:
        datasets["FineWeb"] = analyze_fineweb(SAMPLE_SIZE_FINEWEB)
    except Exception as e:
        print(f"WARNING: FineWeb failed: {e}")

    try:
        datasets["ASDiv"] = analyze_asdiv()
    except Exception as e:
        print(f"WARNING: ASDiv failed: {e}")

    try:
        datasets["ParaMAWPS"] = analyze_paramawps()
    except Exception as e:
        print(f"WARNING: ParaMAWPS failed: {e}")

    try:
        datasets["DMath"] = analyze_dmath()
    except Exception as e:
        print(f"WARNING: DMath failed: {e}")

    try:
        datasets["MathInstruct"] = analyze_mathinstruct(SAMPLE_SIZE_MATHINSTRUCT)
    except Exception as e:
        print(f"WARNING: MathInstruct failed: {e}")

    if datasets:
        summarise(datasets)
        print(
            f"\nEDA complete. Plots saved to:  – {CLOUD_PLOT_DIR.resolve()}"
        )
    else:
        print("No dataset analysis succeeded – nothing to summarise.")


if __name__ == "__main__":
    main()


Loading OpenWebMath …
Loaded 10,000 examples


/var/folders/cw/nsgz30_17pq84g4s1jh0lgtw0000gp/T/ipykernel_79112/478287237.py:108: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_yticklabels(labels, fontsize=TICK_FONTSIZE)



Loading FineWeb …
Loaded 10,000 examples


/var/folders/cw/nsgz30_17pq84g4s1jh0lgtw0000gp/T/ipykernel_79112/478287237.py:108: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_yticklabels(labels, fontsize=TICK_FONTSIZE)



Loading ASDiv …


...


Loading DMath …


/var/folders/cw/nsgz30_17pq84g4s1jh0lgtw0000gp/T/ipykernel_79112/478287237.py:108: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_yticklabels(labels, fontsize=TICK_FONTSIZE)



Loading MathInstruct …
Loaded 10,000 examples


/var/folders/cw/nsgz30_17pq84g4s1jh0lgtw0000gp/T/ipykernel_79112/478287237.py:108: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_yticklabels(labels, fontsize=TICK_FONTSIZE)



=== Token‑count summary ===
     Dataset Field 1 Tokens (Avg) Field 2 Tokens (Avg)  Samples
 OpenWebMath              1568.20                   NA    10000
     FineWeb               596.05                   NA    10000
       ASDiv                34.68                 4.22     2305
   ParaMAWPS                33.24                 4.73    13023
       DMath                34.71                 1.01     7943
MathInstruct                61.86               115.77    10000

EDA complete. Plots saved to:  – /Users/jonathan/Library/Mobile Documents/com~apple~CloudDocs/Master/Master Thesis/math-reasoning-in-language-models/plots/plots_images/eda
